# 🎯 6. Iterators & Iterables

Understanding the **iterator protocol** explains how `for` loops, generators, and many built-in functions work under the hood.

| ☕ Java | 🐍 Python |
|---------|----------|
| `Iterable<T>` interface → `iterator()` | `__iter__()` method |
| `Iterator<T>` interface → `hasNext()` + `next()` | `__next__()` + `StopIteration` |
| Enhanced for-loop uses `Iterable` | `for` loop uses `__iter__` protocol |

## 6.1 Iterable vs Iterator

| Concept | Has | Role |
|---------|-----|------|
| **Iterable** | `__iter__()` | Can create an iterator |
| **Iterator** | `__iter__()` + `__next__()` | Produces values one by one |

Lists, tuples, strings, dicts are **iterables** — the `for` loop calls `iter()` on them to get an iterator.

In [3]:
my_list = list(range(10, 50, 10))

my_iterator = iter(my_list)
print(next(my_iterator))
print(next(my_iterator))
print(next(my_iterator))
print(next(my_iterator))

10
20
30
40


In [4]:
try:
    print(next(my_iterator))
except StopIteration:
    print("Iterator exhausted")

Iterator exhausted


## 6.2 Custom Iterator — Class-based

Implement `__iter__` and `__next__` to make any object iterable.

In [5]:
class Countdown:
    def __init__(self, start: int):
        self.current = start
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value
    
for num in Countdown(5):
    print(f" {num}", end=" ")

 5  4  3  2  1 

In [7]:
print(f"List: {list(Countdown(10))}")
print(f"Sum: {sum(Countdown(10))}")

List: [10, 9, 8, 7, 6, 5, 4, 3, 2, 1]
Sum: 55


## 6.3 Separate Iterable and Iterator (Reusable)

**Problem:** When `__iter__` returns `self`, the object is single-use.
**Solution:** Separate the iterable (factory) from the iterator (state).

In [8]:
class RangeIterator:
    def __init__(self, start: int, end: int):
        self.current = start
        self.end = end
    
    def __iter__(self):
        return self
    
    def __next__(self) -> int:
        if self.current >= self.end:
            raise StopIteration
        value = self.current
        self.current += 1
        return value
    
class Range:
    def __init__(self, start: int, end: int):
        self.start = start
        self.end = end    

    def __iter__(self):
        return RangeIterator(self.start, self.end)    

In [ ]:
my_range = Range(1, 6)

print(f"First call: {list(my_range)}")
print(f"Second call: {list(my_range)}") # Because the Range is iterable object don't print StopIterator Error or None


First call: [1, 2, 3, 4, 5]
Second call: [1, 2, 3, 4, 5]


## 6.6 Real-World: Paginated API Iterator

This is where class-based iterators shine — managing state across pages.

In [ ]:
class PaginatedAPI:
    """Iterator that fetches data page by page.
    Real-world use: paginated REST APIs, database cursors."""
    
    def __init__(self, total_items: int, page_size: int = 3):
        self._total = total_items
        self._page_size = page_size
        self._offset = 0
    
    def __iter__(self):
        self._offset = 0  # Reset for reuse
        return self
    
    def __next__(self) -> list[dict]:
        if self._offset >= self._total:
            raise StopIteration
        
        # Simulate API call
        end = min(self._offset + self._page_size, self._total)
        page = [
            {"id": i, "name": f"Item {i}"}
            for i in range(self._offset, end)
        ]
        print(f"  📡 Fetched page (offset={self._offset}, size={len(page)})")
        self._offset = end
        return page

api = PaginatedAPI(total_items=8, page_size=3)

print("Iterating through all pages:")
for page in api:
    print(f"    {page}")

print("\n💡 Use class-based iterators when you need:")
print("   - Complex state (page tracking, auth tokens)")
print("   - Reusable iteration (reset offset)")
print("   - Custom methods (e.g., .skip_page())")